# 02 — R1CS Basics

**R1CS (Rank-1 Constraint System)** is the standard way to encode a computation as a system of constraints that a prover must satisfy.

## The Idea

Any computation can be broken into a sequence of **multiplication gates**. Each gate takes two inputs and produces one output. We encode the entire computation as:

$$A\mathbf{z} \circ B\mathbf{z} = C\mathbf{z}$$

where:
- $A$, $B$, $C$ are sparse matrices over $\mathbb{F}_p$ (shape: $m \times n$)
- $\mathbf{z} = (1, \mathbf{x}, \mathbf{w})$ concatenates a constant 1, public inputs $\mathbf{x}$, and private witness $\mathbf{w}$
- $\circ$ is the Hadamard (element-wise) product
- Each row is one constraint: $(\mathbf{a}_i \cdot \mathbf{z})(\mathbf{b}_i \cdot \mathbf{z}) = \mathbf{c}_i \cdot \mathbf{z}$

**Nova paper reference:** Section 3 (Preliminaries)

In [ ]:
import numpy as np
from nano_nova.field import GF
from nano_nova.r1cs import R1CSShape

## Example: A Simple Multiplication

Let's encode $y = x_1 \cdot x_2$ where $x_1, x_2$ are public inputs and $y$ is the output.

Variables: $\mathbf{z} = (1, x_1, x_2, y)$, indices 0-3.

One constraint: $x_1 \cdot x_2 = y$
- $A = [0, 1, 0, 0]$ → selects $x_1$
- $B = [0, 0, 1, 0]$ → selects $x_2$  
- $C = [0, 0, 0, 1]$ → selects $y$

In [ ]:
# z = (1, x1, x2, y) — 4 variables, 3 public inputs, 0 witness
m, n, l = 1, 4, 3  # 1 constraint, 4 variables, 3 public inputs

A = GF(np.array([[0, 1, 0, 0]]))  # selects x1
B = GF(np.array([[0, 0, 1, 0]]))  # selects x2
C = GF(np.array([[0, 0, 0, 1]]))  # selects y

shape = R1CSShape(A=A, B=B, C=C, m=m, n=n, l=l)

# With l=3 public inputs and n=4, there's no private witness needed.
# Let's just show the structure — we use Fibonacci (which has a witness) next.
print("Simple multiplication R1CS:")
print(f"A = {A}  (selects x1)")
print(f"B = {B}  (selects x2)")
print(f"C = {C}  (selects y)")
print(f"\nConstraint: (A·z) * (B·z) = (C·z)  →  x1 * x2 = y")

## Example: Fibonacci Step

This is the circuit we use throughout nanoNova. One step of Fibonacci:

$$(a, b) \rightarrow (b, a+b)$$

**Public inputs** ($l=4$): $\mathbf{x} = [a_{in}, b_{in}, a_{out}, b_{out}]$

**Private witness** ($|w|=1$): $\mathbf{w} = [\text{sum}]$ where $\text{sum} = a_{in} + b_{in}$

**Variables:** $\mathbf{z} = (1, a_{in}, b_{in}, a_{out}, b_{out}, \text{sum})$ — indices 0 through 5

**Constraints** ($m=3$):

| # | Meaning | A (left) | B (right) | C (output) |
|---|---------|----------|-----------|------------|
| 1 | $1 \cdot (a_{in} + b_{in}) = \text{sum}$ | $[1,0,0,0,0,0]$ | $[0,1,1,0,0,0]$ | $[0,0,0,0,0,1]$ |
| 2 | $1 \cdot b_{in} = a_{out}$ | $[1,0,0,0,0,0]$ | $[0,0,1,0,0,0]$ | $[0,0,0,1,0,0]$ |
| 3 | $1 \cdot \text{sum} = b_{out}$ | $[1,0,0,0,0,0]$ | $[0,0,0,0,0,1]$ | $[0,0,0,0,1,0]$ |

Note: R1CS is natively *multiplicative*, so addition is encoded as $1 \times (a + b)$.

In [ ]:
from nano_nova.examples import fibonacci_step_circuit, fibonacci_step_fn, fibonacci_witness_fn

shape = fibonacci_step_circuit()
print(f"Fibonacci circuit: {shape.m} constraints, {shape.n} variables, {shape.l} public inputs")
print(f"\nA =\n{shape.A}")
print(f"\nB =\n{shape.B}")
print(f"\nC =\n{shape.C}")

In [ ]:
# Let's trace through one step: (1, 1) -> (1, 2)
z_in = GF(np.array([1, 1]))
z_out = fibonacci_step_fn(z_in)
w = fibonacci_witness_fn(z_in)

print(f"Input:   z_in = {z_in}")   # [1, 1]
print(f"Output:  z_out = {z_out}") # [1, 2]
print(f"Witness: w = {w}")         # [2] (the sum 1+1)

# Public inputs: x = [a_in, b_in, a_out, b_out]
x = GF(np.concatenate([z_in, z_out]))
print(f"\nPublic x = {x}")

# Full variable vector: z = (1, x, w)
z = GF(np.concatenate([GF(np.array([1])), x, w]))
print(f"Full z = {z}")

In [ ]:
# Verify each constraint manually
Az = shape.A @ z
Bz = shape.B @ z
Cz = shape.C @ z

print("Constraint-by-constraint check:")
for i in range(shape.m):
    lhs = Az[i] * Bz[i]  # (a_i · z) * (b_i · z)
    rhs = Cz[i]          # c_i · z
    ok = "✓" if lhs == rhs else "✗"
    print(f"  Constraint {i+1}: {int(Az[i])} * {int(Bz[i])} = {int(lhs)}, expected {int(rhs)} {ok}")

# Or use the built-in check
print(f"\nR1CS satisfied: {shape.is_satisfied(x, w)}")

In [ ]:
# What happens with a WRONG witness?
bad_w = GF(np.array([99]))  # wrong sum
print(f"Wrong witness satisfies R1CS: {shape.is_satisfied(x, bad_w)}")

# What about wrong output?
bad_x = GF(np.array([1, 1, 1, 99]))  # wrong b_out
good_w = GF(np.array([2]))
print(f"Wrong output satisfies R1CS: {shape.is_satisfied(bad_x, good_w)}")

## Multiple Steps

For IVC, we apply the same circuit repeatedly. Each step produces a new $(x_i, w_i)$ pair.

In [ ]:
# Trace 8 Fibonacci steps
z = GF(np.array([1, 1]))
print(f"Step 0: z = ({int(z[0])}, {int(z[1])})")

for i in range(8):
    z_next = fibonacci_step_fn(z)
    w = fibonacci_witness_fn(z)
    x = GF(np.concatenate([z, z_next]))
    ok = shape.is_satisfied(x, w)
    print(f"Step {i+1}: z = ({int(z_next[0]):>4}, {int(z_next[1]):>4})  "
          f"witness={int(w[0]):>4}  R1CS={'✓' if ok else '✗'}")
    z = z_next

## Key Takeaway

R1CS encodes a computation as $A\mathbf{z} \circ B\mathbf{z} = C\mathbf{z}$:
- Each constraint is a **rank-1** equation (product of two linear forms equals a linear form)
- The **witness** $\mathbf{w}$ contains intermediate values the prover knows
- Satisfying the system proves the computation was done correctly

The problem: if we want to prove $f^n(x)$ (n steps), we'd need n separate R1CS proofs. Nova's insight is to **fold** them together.

Next: [03_relaxed_r1cs.ipynb](03_relaxed_r1cs.ipynb) — why naive folding fails and how Relaxed R1CS fixes it.